# NB01 pretrained phoneme eval (Jalur B, zero-shot)
`facebook/wav2vec2-lv-60-espeak-cv-ft`, greedy CTC, no LM, no biasing. Same as the app.

1. PER on own clips (reference = phonemizer + espeak-ng, same as `server.py`).
2. PER on an L2-ARCTIC annotated subset (reference = human perceived phones, ARPAbet mapped to IPA). Kept separate from 1.
3. Timing: T4 fp32, CPU fp32, CPU dynamic int8. Warm-up excluded. RTF = infer time / audio time.

Kaggle CPU is not your laptop CPU. The laptop gate is still `backend/bench_rtf.py`.

Settings: GPU T4, Internet ON. Inputs: `spellspeak-own-clips` (private), optional `l2-arctic` (private, licence CC BY-NC 4.0).

In [ ]:
# Repo code (per.py, words.json). Kaggle cannot see your laptop, so pick one:
#   A) public GitHub repo + Internet ON in notebook settings -> git clone below
#   B) private repo -> upload the repo folder as a private Kaggle Dataset "spellspeak-code" and add it here
import json, os, platform, subprocess, sys, time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

REPO_URL = "https://github.com/ddrlve/SpellSpeak.git"
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
CANDIDATES = [Path("/kaggle/input/spellspeak-code"), Path.cwd().parent, WORK / "SpellSpeak"]
REPO = next((p for p in CANDIDATES if (p / "backend" / "per.py").exists()), None)
if REPO is None:
    REPO = WORK / "SpellSpeak"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO / "backend"))
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO, "| results:", RESULTS)

In [ ]:
def cpu_name() -> str:
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()


def device_info(force_cpu: bool = False) -> dict:
    name, threads = cpu_name(), os.cpu_count()
    try:
        import torch
        threads = torch.get_num_threads()
        if torch.cuda.is_available() and not force_cpu:
            name = torch.cuda.get_device_name(0)
    except ImportError:
        pass
    return {"name": name, "threads": threads, "platform": platform.platform()}


def versions(*pkgs: str) -> dict:
    out = {}
    for p in pkgs:
        try:
            out[p] = metadata.version(p)
        except metadata.PackageNotFoundError:
            out[p] = None
    return out


def write_result(notebook: str, data: dict, metrics: dict, notes: str, pkgs: tuple = ()) -> Path:
    """Shape from docs/NOTEBOOKS.md. Only values computed in this run go into metrics."""
    out = {"notebook": notebook, "date": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "device": device_info(), "data": data, "metrics": metrics, "notes": notes,
           "versions": versions(*pkgs)}
    path = RESULTS / f"{notebook}.json"
    path.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(out, indent=2, ensure_ascii=False))
    return path

In [ ]:
!apt-get -qq update && apt-get -qq install -y espeak-ng > /dev/null
!pip install -q phonemizer

In [ ]:
# Decide BEFORE looking at results (docs/NOTEBOOKS.md step 5). None = report only, no accept/reject verdict.
PER_MARGIN = None          # e.g. absolute PER increase allowed for int8 vs fp32. Dian sets this.
CPU_THREADS = os.cpu_count()
L2_DIR = Path("/kaggle/input/l2-arctic")
L2_MAX_PER_SPEAKER = 20    # annotated utterances per speaker, seeded sample
SEED = 0

In [ ]:
# Own recordings: private Kaggle Dataset made from data/own (backend/record_clips.py output):
#   metadata.csv + wav/*.wav, 16 kHz mono. Real consenting speakers only. Never synthetic clips.
import numpy as np
import pandas as pd
import soundfile as sf

OWN = Path("/kaggle/input/spellspeak-own-clips")
if not (OWN / "metadata.csv").exists():
    raise FileNotFoundError(f"{OWN}/metadata.csv not found. Record clips with backend/record_clips.py, "
                            "upload data/own as a private Kaggle Dataset, add it to this notebook.")
meta = pd.read_csv(OWN / "metadata.csv")
assert meta["consent"].astype(str).str.lower().eq("yes").all(), "every row needs consent=yes"


def load_wav(path: Path) -> np.ndarray:
    audio, sr = sf.read(path, dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000:
        raise ValueError(f"{path}: need 16000 Hz, got {sr}")
    return audio


own_items = [{"id": r.file, "speaker": r.speaker_id, "word": r.word, "audio": load_wav(OWN / r.file)}
             for r in meta.itertuples()]
print(f"own clips: {len(own_items)} | speakers: {meta.speaker_id.nunique()} | words: {meta.word.nunique()}")

In [ ]:
from phonemizer import phonemize
from phonemizer.separator import Separator

_ipa_cache: dict[str, list[str]] = {}


def target_ipa(word: str) -> list[str]:
    """Same call as backend/server.py so the notebook scores exactly what the app scores."""
    if word not in _ipa_cache:
        out = phonemize(word, language="en-us", backend="espeak",
                        separator=Separator(phone=" ", word="", syllable=""), strip=True)
        _ipa_cache[word] = out.split()
    return _ipa_cache[word]

In [ ]:
import copy
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import per as per_mod

MODEL = "facebook/wav2vec2-lv-60-espeak-cv-ft"
processor = Wav2Vec2Processor.from_pretrained(MODEL)
model_fp32 = Wav2Vec2ForCTC.from_pretrained(MODEL).eval()
VOCAB = set(processor.tokenizer.get_vocab())


def recognize(model, audio: np.ndarray, device: torch.device) -> tuple[list[str], float]:
    x = processor(audio, sampling_rate=16000, return_tensors="pt").input_values.to(device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t = time.perf_counter()
    with torch.inference_mode():
        logits = model(x).logits
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t
    return processor.batch_decode(torch.argmax(logits, dim=-1).cpu())[0].split(), elapsed


def run_eval(model, device: torch.device, items: list[dict], tag: str) -> pd.DataFrame:
    model = model.to(device)
    recognize(model, items[0]["audio"], device)  # warm-up, excluded
    rows = []
    for it in items:
        hyp, el = recognize(model, it["audio"], device)
        r = per_mod.per(it["ref"], hyp)
        dur = len(it["audio"]) / 16000
        rows.append({"config": tag, "id": it["id"], "speaker": it["speaker"], "ref": " ".join(it["ref"]),
                     "hyp": " ".join(hyp), "per": r["per"], "S": r["S"], "D": r["D"], "I": r["I"],
                     "N": r["N"], "audio_s": dur, "infer_s": el, "rtf": el / dur})
    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame) -> dict:
    errs = int((df.S + df.D + df.I).sum())
    return {"n": len(df), "speakers": int(df.speaker.nunique()),
            "per_corpus": errs / int(df.N.sum()) if df.N.sum() else None,
            "S": int(df.S.sum()), "D": int(df.D.sum()), "I": int(df.I.sum()), "N": int(df.N.sum()),
            "infer_ms_median": float(df.infer_s.median() * 1000), "infer_ms_p95": float(df.infer_s.quantile(0.95) * 1000),
            "rtf_median": float(df.rtf.median()), "rtf_p95": float(df.rtf.quantile(0.95))}

In [ ]:
# Own clips: reference = phonemizer target of the word the game asked for.
for it in own_items:
    it["ref"] = target_ipa(it["word"])
ref_oov = sorted({p for it in own_items for p in it["ref"] if p not in VOCAB})
print("reference tokens missing from model vocab:", ref_oov)

frames = []
if torch.cuda.is_available():
    frames.append(run_eval(model_fp32, torch.device("cuda"), own_items, "own_t4_fp32"))
torch.set_num_threads(CPU_THREADS)
cpu = torch.device("cpu")
frames.append(run_eval(model_fp32, cpu, own_items, "own_cpu_fp32"))
model_int8 = torch.ao.quantization.quantize_dynamic(copy.deepcopy(model_fp32).cpu(), {torch.nn.Linear}, dtype=torch.qint8)
frames.append(run_eval(model_int8, cpu, own_items, "own_cpu_int8"))
own_df = pd.concat(frames)
own_df.groupby("config").apply(summarize)

## L2-ARCTIC (optional, runs only if the dataset is attached)
Reference = perceived phone from the human annotation (`annotation/*.TextGrid`, tier `phones`).
A label `CPL,PPL,type` means canonical, perceived, error type (s/d/a). Deletions are dropped from the reference.
The ARPAbet to IPA table below is a first draft: check it against the vocab printout and a few files before trusting the PER.
L2-ARCTIC wav is 44.1 kHz, resampled to 16 kHz.

In [ ]:
import random
import re
import torchaudio

ARPA2IPA = {  # first draft, espeak-style en-us. Verify and document in the report.
    "AA": "ɑː", "AE": "æ", "AH": "ʌ", "AH0": "ə", "AO": "ɔː", "AW": "aʊ", "AY": "aɪ", "B": "b", "CH": "tʃ",
    "D": "d", "DH": "ð", "EH": "ɛ", "ER": "ɜː", "ER0": "ɚ", "EY": "eɪ", "F": "f", "G": "ɡ", "HH": "h",
    "IH": "ɪ", "IY": "iː", "JH": "dʒ", "K": "k", "L": "l", "M": "m", "N": "n", "NG": "ŋ", "OW": "oʊ",
    "OY": "ɔɪ", "P": "p", "R": "ɹ", "S": "s", "SH": "ʃ", "T": "t", "TH": "θ", "UH": "ʊ", "UW": "uː",
    "V": "v", "W": "w", "Y": "j", "Z": "z", "ZH": "ʒ",
}
print("mapped IPA not in model vocab:", sorted({v for v in ARPA2IPA.values() if v not in VOCAB}))
SKIP = {"", "sil", "sp", "spn"}


def textgrid_phones(path: Path) -> list[str]:
    text = path.read_text(encoding="utf-8", errors="replace")
    tier = re.search(r'name = "phones".*?(?=item \[\d+\]:|\Z)', text, re.S)
    return re.findall(r'text = "(.*?)"', tier.group(0)) if tier else []


def to_ipa(label: str, unknown: list[str]) -> str | None:
    parts = [p.strip() for p in label.split(",")]
    if len(parts) == 3 and parts[2].lower() == "d":
        return None                               # perceived deletion: not in what was said
    ph = (parts[1] if len(parts) >= 2 else parts[0]).upper().rstrip("*")
    if ph.lower() in SKIP:
        return None
    key = ph if ph in ARPA2IPA else re.sub(r"\d", "", ph)
    if key not in ARPA2IPA:
        unknown.append(ph)
        return None
    return ARPA2IPA[key]


l2_items, unknown = [], []
if L2_DIR.exists():
    rng = random.Random(SEED)
    for spk in sorted(p for p in L2_DIR.iterdir() if (p / "annotation").exists()):
        grids = sorted((spk / "annotation").glob("*.TextGrid"))
        for g in rng.sample(grids, min(L2_MAX_PER_SPEAKER, len(grids))):
            ref = [p for p in (to_ipa(l, unknown) for l in textgrid_phones(g)) if p]
            wav, sr = torchaudio.load(str(spk / "wav" / f"{g.stem}.wav"))
            audio = torchaudio.functional.resample(wav.mean(0), sr, 16000).numpy()
            l2_items.append({"id": f"{spk.name}/{g.stem}", "speaker": spk.name, "ref": ref, "audio": audio})
    print(f"L2-ARCTIC items: {len(l2_items)} | unknown labels skipped: {len(unknown)} {sorted(set(unknown))[:20]}")
else:
    print("L2-ARCTIC not attached, section skipped.")

In [ ]:
l2_df = pd.DataFrame()
if l2_items:
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    l2_df = run_eval(model_fp32, dev, l2_items, "l2arctic_fp32")  # PER only matters here; timing is on that device
    print(summarize(l2_df))

In [ ]:
all_df = pd.concat([own_df, l2_df])
all_df.to_csv(RESULTS / "NB01_per_clip.csv", index=False)
metrics = {cfg: summarize(g) for cfg, g in all_df.groupby("config")}
verdict = None
if PER_MARGIN is not None:
    diff = metrics["own_cpu_int8"]["per_corpus"] - metrics["own_cpu_fp32"]["per_corpus"]
    verdict = {"per_margin": PER_MARGIN, "int8_minus_fp32_per": diff, "accept_int8": diff <= PER_MARGIN}
metrics["int8_verdict"] = verdict
write_result(
    "NB01_pretrained_phoneme_eval",
    data={"source": "own clips" + (" + L2-ARCTIC annotated subset" if l2_items else ""),
          "n_utterances": len(own_items) + len(l2_items), "speakers": int(all_df.speaker.nunique()),
          "own_ref_tokens_not_in_vocab": ref_oov},
    metrics=metrics,
    notes=f"Greedy CTC, no LM. CPU threads={CPU_THREADS}. Kaggle CPU is not the laptop CPU. "
          f"L2 seed={SEED}, max {L2_MAX_PER_SPEAKER}/speaker, unknown labels skipped={len(unknown)}.",
    pkgs=("torch", "torchaudio", "transformers", "phonemizer"),
)